In [1]:
import pandas as pd
import numpy as np
from scipy import stats

# Load data
df = pd.read_csv('Survey.csv')


# Answer Key
cube1_correct = ['8','11','13','27','30']
cube2_correct = ['8','11','16','30','32']
math1_correct = ['75','36','94','98','399']
math2_correct = ['151','36','181','248','368']

def score_round(df, cols, correct):
    total = pd.Series([0] * len(df), index=df.index)
    for col, ans in zip(cols, correct):
        try:
            # Compare numerically to handle '36' vs '36.0' mismatches
            total += (pd.to_numeric(df[col], errors='coerce') == float(ans)).astype(int)
        except ValueError:
            # Fall back to string comparison if answer is not numeric
            total += (df[col].astype(str).str.strip() == ans).astype(int)
    return total

cube1_cols = ['Cube_1_1','Cube_1_2','Cube_1_3','Cube_1_4','Cube_1_5']
cube2_cols = ['Cube_2_1','Cube_2_2','Cube_2_3','Cube_2_4','Cube_2_5']
math1_cols = ['Math_1_1','Math_1_2','Math_1_3','Math_1_4','Math_1_5']
math2_cols = ['Math_2_1','Math_2_2','Math_2_3','Math_2_4','Math_2_5']

df['Cube_R1'] = score_round(df, cube1_cols, cube1_correct)
df['Cube_R2'] = score_round(df, cube2_cols, cube2_correct)
df['Math_R1'] = score_round(df, math1_cols, math1_correct)
df['Math_R2'] = score_round(df, math2_cols, math2_correct)

# Change Scores
df['Mood_Change']   = df['Mood_2']   - df['Mood_1']
df['Energy_Change'] = df['Energy_2'] - df['Energy_1']
df['Math_Change']   = df['Math_R2']  - df['Math_R1']
df['Cube_Change']   = df['Cube_R2']  - df['Cube_R1']

# Groups
con = df[df['Group'] == 'Con']
exp     = df[df['Group'] == 'Exp']

# Formatting 
def fmt(series):
    return f"{series.mean():.1f} ({series.std(ddof=1):.2f})"

# Emotional state mean and SD
print("REPORTED EMOTIONAL STATE")
print(f"{'':6} {'Initial Mood':>14} {'Final Mood':>12} {'Initial Energy':>16} {'Final Energy':>14}")
print(f"{'Con.':6} {fmt(con['Mood_1']):>14} {fmt(con['Mood_2']):>12} {fmt(con['Energy_1']):>16} {fmt(con['Energy_2']):>14}")
print(f"{'Exp.':6} {fmt(exp['Mood_1']):>14} {fmt(exp['Mood_2']):>12} {fmt(exp['Energy_1']):>16} {fmt(exp['Energy_2']):>14}")
print()

# Performance mean and SD
print("CORRECT RESPONSES")
print(f"{'':6} {'Cubes Round 1':>15} {'Cubes Round 2':>15} {'Math Round 1':>14} {'Math Round 2':>14}")
print(f"{'Con.':6} {fmt(con['Cube_R1']):>15} {fmt(con['Cube_R2']):>15} {fmt(con['Math_R1']):>14} {fmt(con['Math_R2']):>14}")
print(f"{'Exp.':6} {fmt(exp['Cube_R1']):>15} {fmt(exp['Cube_R2']):>15} {fmt(exp['Math_R1']):>14} {fmt(exp['Math_R2']):>14}")
print()


# Cohen's d
def cohens_d(a, b):
    pooled_std = np.sqrt((np.std(a, ddof=1)**2 + np.std(b, ddof=1)**2) / 2)
    return (np.mean(a) - np.mean(b)) / pooled_std if pooled_std != 0 else 0

# Mann-Whitney U Test
# scipy.stats.mannwhitneyu handles everything: U, z (via normal approximation), and p
metrics = [
    ('Mood_Change',   'Mood Change'),
    ('Energy_Change', 'Energy Change'),
    ('Math_Change',   'Math Score Change'),
    ('Cube_Change',   'Cube Score Change'),
]

print(f"{'Metric':<22} {'U':>6} {'p':>6} {'d':>7}  Direction")
print("-" * 60)

for col, label in metrics:
    a = exp[col].dropna()
    b = con[col].dropna()
    
    # Mann-Whitney U — use alternative='two-sided' for a two-tailed test
    # method='asymptotic' uses the normal approximation to get a z-based p-value
    result = stats.mannwhitneyu(a, b, alternative='two-sided', method='asymptotic')
    
    d = cohens_d(a, b)
    direction = "favoring experiment" if np.mean(a) > np.mean(b) else "favoring control"
    
    print(f"{label:<22} {result.statistic:>6.1f} {result.pvalue:>6.3f} {d:>7.3f}  {direction}")

# Normaloty Check (Shapiro-Wilk)
print("\nShapiro-Wilk Normality Tests:")
print(f"{'Metric':<22} {'Control W':>10} {'Control p':>10} {'Exp W':>8} {'Exp p':>8}")
print("-" * 62)

for col, label in metrics:
    c_vals = con[col].dropna()
    e_vals = exp[col].dropna()
    w_c, p_c = stats.shapiro(c_vals)
    w_e, p_e = stats.shapiro(e_vals)
    print(f"{label:<22} {w_c:>10.3f} {p_c:>10.3f} {w_e:>8.3f} {p_e:>8.3f}")

# Baseline Equivalence
print("\nBaseline Equivalence (Round 1):")
baseline_metrics = [
    ('Mood_1',   'Mood'),
    ('Energy_1', 'Energy'),
    ('Math_R1',  'Math Score'),
    ('Cube_R1',  'Cube Score'),
]
print(f"{'Metric':<22} {'U':>6} {'p':>6}")
print("-" * 36)

for col, label in baseline_metrics:
    a = exp[col].dropna()
    b = con[col].dropna()
    result = stats.mannwhitneyu(a, b, alternative='two-sided', method='asymptotic')
    print(f"{label:<22} {result.statistic:>6.1f} {result.pvalue:>6.3f}")

REPORTED EMOTIONAL STATE
         Initial Mood   Final Mood   Initial Energy   Final Energy
Con.       4.7 (1.16)   4.8 (1.14)       4.1 (1.45)     4.7 (1.34)
Exp.       5.3 (0.67)   5.3 (0.48)       4.5 (1.08)     4.7 (0.95)

CORRECT RESPONSES
         Cubes Round 1   Cubes Round 2   Math Round 1   Math Round 2
Con.        4.8 (0.42)      4.4 (0.70)     3.7 (0.95)     4.3 (0.67)
Exp.        4.7 (0.48)      4.6 (0.70)     3.6 (0.70)     4.5 (0.97)

Metric                      U      p       d  Direction
------------------------------------------------------------
Mood Change              46.0  0.756  -0.118  favoring control
Energy Change            37.0  0.281  -0.537  favoring control
Math Score Change        61.5  0.386   0.264  favoring experiment
Cube Score Change        60.5  0.405   0.349  favoring experiment

Shapiro-Wilk Normality Tests:
Metric                  Control W  Control p    Exp W    Exp p
--------------------------------------------------------------
Mood Change    